# Notebook 05 — Error Analysis & Model Comparison

**Purpose**:
1. Compare all models on both tasks
2. Analyze errors (binary + type)
3. Generate final comparison report

**Prerequisite**: Run notebooks 01–04 first.

**Outputs**:
- `outputs/reports/model_comparison.md`
- `outputs/reports/error_analysis.md`
- `outputs/reports/error_examples_binary.csv`
- `outputs/reports/error_examples_type.csv`

In [ ]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Robust project-root finder ─────────────────────────────────────────────
def _find_project_root() -> Path:
    """Walk up from cwd until we find outputs/splits or another project marker."""
    markers = [
        Path("outputs") / "splits",
        Path("outputs") / "datasets",
        Path("data") / "processed",
        Path("notebooks"),
    ]
    for root in [Path.cwd()] + list(Path.cwd().parents):
        for marker in markers:
            if (root / marker).exists():
                return root
    return Path.cwd().parent  # fallback

ROOT        = _find_project_root()
CLASSICAL   = ROOT / "outputs" / "classical"
BERT_OUT    = ROOT / "outputs" / "bert"
REPORTS_DIR = ROOT / "outputs" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root     : {ROOT}")
print(f"Reports directory: {REPORTS_DIR}")


## 1. Load All Metrics

In [ ]:
def load_metrics(path: Path) -> dict | None:
    if path.exists():
        with open(path) as f:
            return json.load(f)
    print(f"  [WARNING] Not found: {path.relative_to(ROOT)}")
    return None


# All metrics
tfidf_lr_binary = load_metrics(CLASSICAL / "tfidf_lr" / "metrics_binary.json")
tfidf_lr_type   = load_metrics(CLASSICAL / "tfidf_lr" / "metrics_type.json")
nb_binary       = load_metrics(CLASSICAL / "naive_bayes" / "metrics_binary.json")
nb_type         = load_metrics(CLASSICAL / "naive_bayes" / "metrics_type.json")
distilbert_bin  = load_metrics(BERT_OUT / "distilbert_binary" / "metrics.json")
distilbert_type = load_metrics(BERT_OUT / "distilbert_type"   / "metrics.json")
bert_base_bin   = load_metrics(BERT_OUT / "bert_base_binary"  / "metrics.json")  # optional
bert_base_type  = load_metrics(BERT_OUT / "bert_base_type"    / "metrics.json")  # optional

print("Metrics loaded (None = not yet run):")
for name, m in [("TF-IDF+LR binary", tfidf_lr_binary), ("TF-IDF+LR type", tfidf_lr_type),
                ("NB binary", nb_binary), ("NB type", nb_type),
                ("DistilBERT binary", distilbert_bin), ("DistilBERT type", distilbert_type),
                ("BERT-base binary", bert_base_bin), ("BERT-base type", bert_base_type)]:
    print(f"  {name}: {'âœ“' if m else 'âœ—'}")

## 2. Model Comparison Tables

In [ ]:
def extract_test_metrics(metrics_dict: dict | None, split: str = "test") -> dict:
    """Extract test-split metrics from a metrics dict, handling different formats."""
    if metrics_dict is None:
        return {"accuracy": None, "f1_macro": None, "f1_weighted": None,
                "precision_macro": None, "recall_macro": None}
    test_m = metrics_dict.get(split, metrics_dict)  # BERT format uses 'test' key directly
    return {
        "accuracy"        : test_m.get("accuracy"),
        "f1_macro"        : test_m.get("f1_macro"),
        "f1_weighted"     : test_m.get("f1_weighted"),
        "precision_macro" : test_m.get("precision_macro"),
        "recall_macro"    : test_m.get("recall_macro"),
    }


binary_rows = []
type_rows   = []

model_map_bin = [
    ("TF-IDF + LR",   tfidf_lr_binary),
    ("Naive Bayes",   nb_binary),
    ("DistilBERT",    distilbert_bin),
    ("BERT-base",     bert_base_bin),
]

model_map_type = [
    ("TF-IDF + LR",   tfidf_lr_type),
    ("Naive Bayes",   nb_type),
    ("DistilBERT",    distilbert_type),
    ("BERT-base",     bert_base_type),
]

for name, m in model_map_bin:
    row = {"Model": name}
    row.update(extract_test_metrics(m))
    binary_rows.append(row)

for name, m in model_map_type:
    row = {"Model": name}
    row.update(extract_test_metrics(m))
    type_rows.append(row)

binary_comp = pd.DataFrame(binary_rows).set_index("Model")
type_comp   = pd.DataFrame(type_rows).set_index("Model")

print("=== BINARY TASK â€” Test Metrics ===")
print(binary_comp.to_string(float_format=lambda x: f"{x:.4f}" if x is not None else "N/A"))
print()
print("=== TYPE TASK â€” Test Metrics ===")
print(type_comp.to_string(float_format=lambda x: f"{x:.4f}" if x is not None else "N/A"))

In [ ]:
# â”€â”€ Comparison bar charts â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, comp, title in [
    (axes[0], binary_comp, "Binary Task"),
    (axes[1], type_comp,   "Type Task"),
]:
    valid = comp.dropna(subset=["f1_macro"])
    if len(valid) == 0:
        ax.set_title(f"{title} â€” No data")
        continue
    models  = valid.index.tolist()
    f1_vals = valid["f1_macro"].tolist()
    acc_vals = valid["accuracy"].tolist()
    x = range(len(models))
    w = 0.35
    bars1 = ax.bar([i - w/2 for i in x], f1_vals,  w, label="Macro-F1",  color="steelblue")
    bars2 = ax.bar([i + w/2 for i in x], acc_vals, w, label="Accuracy",  color="coral")
    ax.set_xticks(list(x)); ax.set_xticklabels(models, rotation=20, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(f"{title} â€” Model Comparison (Test)", fontsize=13)
    ax.legend()
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{bar.get_height():.3f}", ha="center", fontsize=8)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{bar.get_height():.3f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig(REPORTS_DIR / "model_comparison_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/reports/model_comparison_chart.png")

## 3. Error Analysis â€” Binary Task

In [ ]:
def load_predictions(path: Path) -> pd.DataFrame | None:
    if path.exists():
        return pd.read_csv(path)
    print(f"  [WARNING] Not found: {path}")
    return None


# Use best available binary model predictions for error analysis
# Priority: BERT > TF-IDF+LR > NB
pred_paths_bin = [
    ("DistilBERT",  BERT_OUT / "distilbert_binary" / "predictions_test.csv"),
    ("TF-IDF + LR", CLASSICAL / "tfidf_lr" / "predictions_test_binary.csv"),
    ("Naive Bayes", CLASSICAL / "naive_bayes" / "predictions_test_binary.csv"),
]

error_source_bin = None
error_model_bin  = None
for model_name, path in pred_paths_bin:
    df = load_predictions(path)
    if df is not None:
        error_source_bin = df
        error_model_bin  = model_name
        print(f"Using {model_name} predictions for binary error analysis")
        break

if error_source_bin is None:
    print("No binary predictions found. Run at least one model first.")

In [ ]:
if error_source_bin is not None:
    pred_col = "predicted" if "predicted" in error_source_bin.columns else "predicted_id"
    true_col = "binary_label"
    err_bin  = error_source_bin[error_source_bin["correct"] == 0].copy()

    # Categorize
    fp_bin = err_bin[err_bin[true_col] == 0].copy()  # False Positives (predicted sarcastic)
    fn_bin = err_bin[err_bin[true_col] == 1].copy()  # False Negatives (predicted not-sarcastic)

    print(f"Total binary test errors: {len(err_bin)}")
    print(f"  False Positives (non-sarcastic predicted as sarcastic) : {len(fp_bin)}")
    print(f"  False Negatives (sarcastic predicted as non-sarcastic)  : {len(fn_bin)}")

    # Select 20+ errors: mix of FP and FN
    n_each = 12
    sample_fp = fp_bin.head(n_each)
    sample_fn = fn_bin.head(n_each)
    error_sample_bin = pd.concat([sample_fp, sample_fn], ignore_index=True)
    error_sample_bin["error_type"] = ["FP"] * len(sample_fp) + ["FN"] * len(sample_fn)

    print(f"\nError sample size: {len(error_sample_bin)} examples")
    print("\n--- False Positives (non-sarcastic â†’ predicted sarcastic) ---")
    for _, row in sample_fp.iterrows():
        print(f"  {row['text']}")

    print("\n--- False Negatives (sarcastic â†’ predicted non-sarcastic) ---")
    for _, row in sample_fn.iterrows():
        print(f"  {row['text']}")

In [ ]:
if error_source_bin is not None:
    # Failure mode categorization (heuristic rules)
    def categorize_binary_error(text: str, error_type: str) -> str:
        text_lower = text.lower()
        if "?" in text and error_type == "FP":
            return "rhetorical_phrasing"
        if any(w in text_lower for w in ["report", "study", "survey", "finds", "shows"]):
            return "neutral_reporting_style_confused"
        if any(w in text_lower for w in ["best", "great", "amazing", "wonderful", "perfect"]):
            return "positive_framing_confused"
        if "onion" in text_lower:
            return "domain_leak"
        if len(text.split()) <= 5:
            return "very_short_text"
        return "lexical_ambiguity"

    error_sample_bin["failure_mode"] = error_sample_bin.apply(
        lambda r: categorize_binary_error(r["text"], r["error_type"]), axis=1
    )

    print("Failure mode distribution:")
    print(error_sample_bin["failure_mode"].value_counts())

    # Save
    error_sample_bin.to_csv(REPORTS_DIR / "error_examples_binary.csv", index=False)
    print("\nSaved: outputs/reports/error_examples_binary.csv")

## 4. Error Analysis â€” Type Task

In [ ]:
pred_paths_type = [
    ("DistilBERT",  BERT_OUT / "distilbert_type" / "predictions_test.csv"),
    ("TF-IDF + LR", CLASSICAL / "tfidf_lr" / "predictions_test_type.csv"),
    ("Naive Bayes", CLASSICAL / "naive_bayes" / "predictions_test_type.csv"),
]

error_source_type = None
error_model_type  = None
for model_name, path in pred_paths_type:
    df = load_predictions(path)
    if df is not None:
        error_source_type = df
        error_model_type  = model_name
        print(f"Using {model_name} predictions for type error analysis")
        break

if error_source_type is None:
    print("No type predictions found.")

In [ ]:
if error_source_type is not None:
    err_type = error_source_type[error_source_type["correct"] == 0].copy()

    # Identify label columns
    true_col_type = "type_label"
    pred_col_type = "predicted_label" if "predicted_label" in err_type.columns else "predicted"

    print(f"Total type errors: {len(err_type)}")
    print("\nTop confusion pairs (true â†’ predicted):")
    if pred_col_type in err_type.columns:
        pairs = err_type.groupby([true_col_type, pred_col_type]).size().sort_values(ascending=False).head(15)
        print(pairs.to_string())

    # Sample 20+ errors
    error_sample_type = err_type.sample(min(25, len(err_type)), random_state=42)

    print(f"\nError sample size: {len(error_sample_type)} examples")
    print("\n--- Sample type misclassifications ---")
    display_cols = ["text", true_col_type]
    if pred_col_type in error_sample_type.columns:
        display_cols.append(pred_col_type)
    for _, row in error_sample_type.head(15).iterrows():
        true_l = row[true_col_type]
        pred_l = row.get(pred_col_type, "?")
        print(f"  [True={true_l:20s}  Pred={pred_l:20s}] {row['text'][:90]}")

In [ ]:
if error_source_type is not None:
    def categorize_type_error(text: str, true_label: str, pred_label: str) -> str:
        """Heuristic failure mode categorization for type task."""
        text_lower = text.lower()
        # Strategy overlap: sarcasm â†” irony â†” satire are frequently confused
        overlap_pairs = {
            frozenset({"sarcasm", "irony"}),
            frozenset({"sarcasm", "satire"}),
            frozenset({"irony", "satire"}),
        }
        if frozenset({true_label, pred_label}) in overlap_pairs:
            return "strategy_semantic_overlap"
        if "rhetorical_question" in [true_label, pred_label]:
            if "?" in text:
                return "rhetorical_vs_other_with_question"
            return "rhetorical_without_question_mark"
        if true_label in ["overstatement", "understatement"] and pred_label in ["sarcasm", "irony"]:
            return "scale_confusion_with_sarcasm"
        if "report" in text_lower or "according" in text_lower:
            return "generation_artifact_formal_phrasing"
        return "lexical_ambiguity"

    if pred_col_type in error_sample_type.columns:
        error_sample_type["failure_mode"] = error_sample_type.apply(
            lambda r: categorize_type_error(r["text"], r[true_col_type], r[pred_col_type]),
            axis=1
        )
        print("Failure mode distribution:")
        print(error_sample_type["failure_mode"].value_counts())

    error_sample_type.to_csv(REPORTS_DIR / "error_examples_type.csv", index=False)
    print("\nSaved: outputs/reports/error_examples_type.csv")

## 5. Generate Final Reports

In [ ]:
def fmt(v) -> str:
    """Format a metric value for markdown."""
    if v is None:
        return "N/A"
    return f"{v:.4f}"


def build_model_comparison_md() -> str:
    lines = []
    lines.append("# Model Comparison Report")
    lines.append("")
    lines.append("> Auto-generated from outputs of notebooks 02â€“04.")
    lines.append("")

    # Binary task table
    lines.append("## Task A â€” Binary Classification (Test Set)")
    lines.append("")
    lines.append("| Model | Accuracy | Precision (M) | Recall (M) | F1 (Macro) | F1 (Weighted) |")
    lines.append("|-------|----------|--------------|------------|------------|--------------|")
    for name, m in model_map_bin:
        r = extract_test_metrics(m)
        lines.append(
            f"| {name} | {fmt(r['accuracy'])} | {fmt(r['precision_macro'])} | "
            f"{fmt(r['recall_macro'])} | {fmt(r['f1_macro'])} | {fmt(r['f1_weighted'])} |"
        )
    lines.append("")
    lines.append("_Primary metric: Macro-F1. Best value bolded (manually after review)._")
    lines.append("")

    # Type task table
    lines.append("## Task B â€” Sarcasm Type Classification (Test Set)")
    lines.append("")
    lines.append("| Model | Accuracy | Precision (M) | Recall (M) | F1 (Macro) | F1 (Weighted) |")
    lines.append("|-------|----------|--------------|------------|------------|--------------|")
    for name, m in model_map_type:
        r = extract_test_metrics(m)
        lines.append(
            f"| {name} | {fmt(r['accuracy'])} | {fmt(r['precision_macro'])} | "
            f"{fmt(r['recall_macro'])} | {fmt(r['f1_macro'])} | {fmt(r['f1_weighted'])} |"
        )
    lines.append("")

    # Recommendation
    lines.append("## Recommendation")
    lines.append("")

    # Find best binary model
    best_bin_name, best_bin_f1 = "N/A", -1
    for name, m in model_map_bin:
        r = extract_test_metrics(m)
        if r["f1_macro"] is not None and r["f1_macro"] > best_bin_f1:
            best_bin_f1 = r["f1_macro"]
            best_bin_name = name

    best_type_name, best_type_f1 = "N/A", -1
    for name, m in model_map_type:
        r = extract_test_metrics(m)
        if r["f1_macro"] is not None and r["f1_macro"] > best_type_f1:
            best_type_f1 = r["f1_macro"]
            best_type_name = name

    lines.append(f"### Best model for Binary Task: **{best_bin_name}** (Macro-F1 = {fmt(best_bin_f1)})")
    lines.append("")
    lines.append(f"### Best model for Type Task: **{best_type_name}** (Macro-F1 = {fmt(best_type_f1)})")
    lines.append("")
    lines.append("### Trade-offs")
    lines.append("")
    lines.append("| Aspect | TF-IDF + LR | Naive Bayes | DistilBERT |")
    lines.append("|--------|------------|-------------|-----------|")
    lines.append("| Training speed | Fast (minutes) | Very fast (seconds) | Slow (hours) |")
    lines.append("| Inference speed | Very fast | Very fast | Moderate |")
    lines.append("| Interpretability | High (feature weights) | High (log-probs) | Low (black-box) |")
    lines.append("| Handles context | No | No | Yes (self-attention) |")
    lines.append("| Handles rare words | Via TF-IDF | Via smoothing | Via sub-word tokenization |")
    lines.append("| GPU required | No | No | Recommended |")
    lines.append("")
    lines.append("**Deployment recommendation**: Use TF-IDF+LR if real-time speed and interpretability are priorities.")
    lines.append("Use DistilBERT for maximum accuracy when GPU inference is available.")

    return "\n".join(lines)


comparison_md = build_model_comparison_md()
with open(REPORTS_DIR / "model_comparison.md", "w", encoding="utf-8") as f:
    f.write(comparison_md)

print("Saved: outputs/reports/model_comparison.md")
print("\nPreview:")
print(comparison_md[:2000])

In [ ]:
def build_error_analysis_md() -> str:
    lines = []
    lines.append("# Error Analysis Report")
    lines.append("")
    lines.append("> Auto-generated from model predictions. Examples drawn from test set.")
    lines.append("")

    # â”€â”€ Binary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    lines.append("## Task A â€” Binary Classification Errors")
    lines.append("")

    if error_source_bin is not None:
        err_df = error_source_bin[error_source_bin["correct"] == 0]
        n_fp = len(err_df[err_df["binary_label"] == 0])
        n_fn = len(err_df[err_df["binary_label"] == 1])
        lines.append(f"**Model**: {error_model_bin}  |  **Total errors**: {len(err_df):,}  |  FP: {n_fp:,}  |  FN: {n_fn:,}")
        lines.append("")
        lines.append("### Failure Mode Summary")
        lines.append("")
        lines.append("| Failure Mode | Description |")
        lines.append("|-------------|-------------|")
        lines.append("| Lexical ambiguity | Sarcasm/irony requires pragmatic context beyond lexical cues |")
        lines.append("| Neutral reporting style confused | Formal generated text mimics neutral news, but still classified as sarcastic |")
        lines.append("| Positive framing confused | Genuine positive news shares superlative vocabulary with ironic praise |")
        lines.append("| Rhetorical phrasing | Questions that look rhetorical but are literal |")
        lines.append("| Very short text | Too little context for confident classification |")
        lines.append("")
        lines.append("### False Positive Examples (Non-sarcastic predicted as sarcastic)")
        lines.append("")
        fp_examples = error_source_bin[
            (error_source_bin["correct"] == 0) & (error_source_bin["binary_label"] == 0)
        ].head(10)
        for _, row in fp_examples.iterrows():
            lines.append(f"- `{row['text']}`")
        lines.append("")
        lines.append("### False Negative Examples (Sarcastic predicted as non-sarcastic)")
        lines.append("")
        fn_examples = error_source_bin[
            (error_source_bin["correct"] == 0) & (error_source_bin["binary_label"] == 1)
        ].head(10)
        for _, row in fn_examples.iterrows():
            lines.append(f"- `{row['text']}`")
    else:
        lines.append("_Binary predictions not available. Run at least one model._")

    lines.append("")

    # â”€â”€ Type â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    lines.append("## Task B â€” Sarcasm Type Classification Errors")
    lines.append("")

    if error_source_type is not None:
        err_df_t = error_source_type[error_source_type["correct"] == 0]
        lines.append(f"**Model**: {error_model_type}  |  **Total errors**: {len(err_df_t):,}")
        lines.append("")
        lines.append("### Failure Mode Summary")
        lines.append("")
        lines.append("| Failure Mode | Description |")
        lines.append("|-------------|-------------|")
        lines.append("| Strategy semantic overlap | sarcasm/irony/satire share similar surface forms; labels conflated |")
        lines.append("| Scale confusion | overstatement/understatement confused with sarcasm due to exaggeration cues |")
        lines.append("| Rhetorical vs. other | rhetorical_question confused with irony/sarcasm when ? is absent |")
        lines.append("| Generation artifact | formal paraphrase style shifts text away from original strategy markers |")
        lines.append("| Lexical ambiguity | strategy relies on world knowledge rather than surface vocabulary |")
        lines.append("")
        lines.append("### Key Insight")
        lines.append("")
        lines.append("The primary failure mode is **strategy semantic overlap**: sarcasm, irony, and satire")
        lines.append("are conceptually related and frequently share similar linguistic surface patterns.")
        lines.append("The label `sarcasm` as a strategy within a sarcasm dataset introduces circularity.")
        lines.append("The `rhetorical_question` class is syntactically distinctive (ends with ?) and should")
        lines.append("be easier to classify; errors in this class suggest the classifier may ignore punctuation.")
        lines.append("")
        lines.append("### Sample Misclassifications")
        lines.append("")
        sample_t = error_source_type[error_source_type["correct"] == 0].head(20)
        true_c = "type_label"
        pred_c = "predicted_label" if "predicted_label" in sample_t.columns else "predicted"
        for _, row in sample_t.iterrows():
            true_l = row[true_c]
            pred_l = row.get(pred_c, "?")
            lines.append(f"- **True**: {true_l} | **Pred**: {pred_l} | `{row['text'][:100]}`")
    else:
        lines.append("_Type predictions not available. Run at least one model._")

    lines.append("")
    lines.append("## Summary of Observations")
    lines.append("")
    lines.append("1. **Binary task** is relatively tractable â€” TheOnion writing style has strong lexical signatures")
    lines.append("2. **Generated headlines** (is_generated=1) may fool classifiers trained mainly on original text")
    lines.append("3. **Type task** is fundamentally harder because strategies are not mutually exclusive")
    lines.append("4. **Class imbalance** (rhetorical_question = 3.7%) is a significant challenge")
    lines.append("5. **BERT** should better capture contextual incongruence; classical models rely on surface vocabulary")

    return "\n".join(lines)


error_md = build_error_analysis_md()
with open(REPORTS_DIR / "error_analysis.md", "w", encoding="utf-8") as f:
    f.write(error_md)

print("Saved: outputs/reports/error_analysis.md")

In [ ]:
print("====== ERROR ANALYSIS COMPLETE ======")
print()
print("Saved artifacts:")
for p in sorted(REPORTS_DIR.iterdir()):
    print(f"  {p.relative_to(ROOT)}")
print()
print("Pipeline complete. See outputs/reports/model_comparison.md for final results.")